# Testing BAZTA Trust Engine with Machine Learning Anomalies

This notebook tests the behavior of the `TrustEngine` under various traffic scenarios, specifically focusing on **ML-detected anomalies that do NOT trigger the hardcoded heuristic rules**.

### Heuristic Rules Reference:
- **ICMP Flood**: `pkt_rate > 90` (Penalty: -50)
- **Port Scan**: `unique_ports > 20` (Penalty: -40)
- **High Entropy**: `port_entropy > 3.5` (Penalty: -20)
- **Byte Flood**: `byte_rate > 50,000` (Penalty: -30)

In [ ]:
import sys
import os
import numpy as np

# Add core path so we can import TrustEngine
sys.path.append(os.path.abspath('../core'))
from trust_engine import TrustEngine

# Initialize TrustEngine (loads models/live_if_model.pkl and models/live_scaler.pkl)
engine = TrustEngine(models_dir=".")
print("TrustEngine successfully initialized!")

## 1. Benign Traffic Test
We start by simulating benign traffic. The trust score should remain high (or recover back to `100.0` if it was lower).

In [ ]:
engine.reset()
benign_flow = {
    "src_ip": "10.0.0.5",
    "pkt_rate": 10.0,
    "byte_rate": 1500.0,
    "proto": 6.0, # TCP
    "pkt_size_variance": 500.0,
    "unique_ports": 2,
    "port_entropy": 0.5
}

res = engine.update(benign_flow)
print("Benign Traffic Result:")
print(res)

## 2. Standard Heuristic Attack (For Comparison)
This represents a traditional attack (e.g. an ICMP flood) that triggers both heuristic rules and ML anomalies.

In [ ]:
engine.reset()
heuristic_attack = {
    "src_ip": "10.0.0.6",
    "pkt_rate": 700.0,       # Triggers icmp_flood (>90)
    "byte_rate": 60000.0,     # Triggers byte_flood (>50k)
    "proto": 6.0,
    "pkt_size_variance": 200.0,
    "unique_ports": 1,
    "port_entropy": 0.0
}

res = engine.update(heuristic_attack)
print("Heuristic Attack Result:")
print(res)

## 3. New ML Anomalies (Bypassing Heuristic Rules)
Here we test scenarios that do **NOT** exceed any of the heuristic thresholds, but are detected as anomalous by the **Isolation Forest** model.

### Case A: Stealthy Flooding (Medium-Rate Attack)
The packet rate is `80` (below the limit of 90) and the byte rate is `45,000` (below the limit of 50,000). No heuristics should trigger, but the ML model should flag it as an anomaly due to the high density relative to normal traffic.

In [ ]:
engine.reset()
stealthy_flow = {
    "src_ip": "10.0.0.11",
    "pkt_rate": 80.0,            # below heuristic 90
    "byte_rate": 45000.0,        # below heuristic 50000
    "proto": 6.0,
    "pkt_size_variance": 100.0,
    "unique_ports": 1,
    "port_entropy": 0.0
}

res = engine.update(stealthy_flow)
print("Stealthy Flooding Result:")
print(res)

### Case B: Covert Channel (Extreme Packet Size Variance)
A very slow flow (packet rate of `15`, byte rate of `5,000`), but with massive packet size variance (`900,000`). This alternates between small control packets and very large packets, indicating potential exfiltration or tunneling.

In [ ]:
engine.reset()
covert_channel_flow = {
    "src_ip": "10.0.0.12",
    "pkt_rate": 15.0,            # low
    "byte_rate": 5000.0,         # low
    "proto": 17.0,               # UDP
    "pkt_size_variance": 900000.0, # extremely high variance
    "unique_ports": 1,
    "port_entropy": 0.0
}

res = engine.update(covert_channel_flow)
print("Covert Channel Result:")
print(res)

### Case C: Atypical Protocol Communication
A device sending a tiny amount of traffic (packet rate of `5`, byte rate of `400`), but using an unusual protocol type (e.g. GRE protocol = `47.0`). Benign traffic is mostly TCP/UDP, so the ML model should flag this unusual protocol as anomalous.

In [ ]:
engine.reset()
atypical_proto_flow = {
    "src_ip": "10.0.0.13",
    "pkt_rate": 5.0,             # low
    "byte_rate": 400.0,          # low
    "proto": 47.0,               # GRE protocol
    "pkt_size_variance": 10.0,
    "unique_ports": 1,
    "port_entropy": 0.0
}

res = engine.update(atypical_proto_flow)
print("Atypical Protocol Result:")
print(res)

### Case D: High-Payload Exfiltration (Below Heuristics)
A device transferring large packets at a moderate rate. `pkt_rate` is `35` (well below 90) and `byte_rate` is `48,000` (below 50,000). The combination of a relatively low packet rate but large throughput and a high size variance (`50,000`) triggers the ML model.

In [ ]:
engine.reset()
exfiltration_flow = {
    "src_ip": "10.0.0.14",
    "pkt_rate": 35.0,            # below 90
    "byte_rate": 48000.0,        # below 50000
    "proto": 6.0,
    "pkt_size_variance": 50000.0,
    "unique_ports": 2,
    "port_entropy": 0.4
}

res = engine.update(exfiltration_flow)
print("High-Payload Exfiltration Result:")
print(res)

## 4. Trust Score Recovery Test
We verify that if a device triggers an anomaly (dropping its score), but later resumes benign behavior, its trust score recovers towards 100 gradually.

In [ ]:
engine.reset()
host = "10.0.0.20"

# 1. Inject an anomaly (Case A)
anomaly = {
    "src_ip": host,
    "pkt_rate": 80.0,
    "byte_rate": 45000.0,
    "proto": 6.0,
    "pkt_size_variance": 100.0,
    "unique_ports": 1,
    "port_entropy": 0.0
}
print("Step 1: Traffic is anomalous")
print(engine.update(anomaly))

# 2. Send benign traffic in consecutive windows
benign = {
    "src_ip": host,
    "pkt_rate": 10.0,
    "byte_rate": 1500.0,
    "proto": 6.0,
    "pkt_size_variance": 500.0,
    "unique_ports": 2,
    "port_entropy": 0.5
}

print("\nStep 2: Resuming normal traffic (monitoring recovery over time)")
for i in range(1, 6):
    res = engine.update(benign)
    print(f"Window {i} trust score: {res['trust_score']} (Action: {res['action']})")